In [26]:
import os
from dotenv import load_dotenv
from google import genai
from openai import OpenAI

load_dotenv()

# Initialize clients
google_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

# Priority rotation list: (provider, model_name)
MODEL_PIPELINE = [
    # ==========================================
    # Groq (Free Developer Tier)
    # Note: No credit card required for free tier.
    # Limits: ~30 RPM, 6,000 TPM, 14,400 RPD.
    # ==========================================
    ("groq", "openai/gpt-oss-20b"),
    ("groq", "openai/gpt-oss-120b"),
    ("groq", "qwen/qwen3.6-27b"),
    ("groq", "qwen/qwen3.8-27b"),
    ("groq", "groq/compound"),
    # ==========================================
    # OpenRouter (Free Tier)
    # Note: The ":free" suffix is MANDATORY. 
    # OpenRouter rotates these based on sponsorships.
    # ==========================================
    ("openrouter", "nvidia/nemotron-3-super:free"),
    ("openrouter", "nvidia/nemotron-3.5-lightning:free"),
    ("openrouter", "google/gemma-4-26b-a4b:free"),
    ("openrouter", "google/gemma-4-31b:free"),
    ("openrouter", "thinkingmachines/inkling:free"),
    
    # ==========================================
    # Google GenAI (Free Tier)
    # Note: These models show "$0.00" for input/output 
    # on the Free Tier, but are strictly rate-limited 
    # (e.g., 15-60 RPM, 1M tokens/day) and data 
    # may be used to improve Google's products.
    # ==========================================
    ("google", "gemini-3.6-flash"),
    ("google", "gemini-3.8-flash"),
    ("google", "gemini-3.7-flash"),
    ("google", "gemini-3.5-flash"),
    ("google", "gemini-3-flash"),
    ("google", "gemini-3.5-flash-lite"),
    ("google", "gemini-2.5-flash"),
    ("google", "gemini-2.5-flash-lite"),
    ("google", "gemma-4"), # Open model, free via API
]

def generate_with_fallback(prompt: str):
    for provider, model in MODEL_PIPELINE:
        try:
            print(f"Attempting model: {provider} -> {model}...")
            
            if provider == "google":
                res = google_client.models.generate_content(
                    model=model,
                    contents=prompt
                )
                return res.text

            elif provider == "groq":
                res = groq_client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}]
                )
                return res.choices[0].message.content

            elif provider == "openrouter":
                res = openrouter_client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}]
                )
                return res.choices[0].message.content

        except Exception as e:
            print(f"Failed on {model} ({e}). Switching to next model...")

    raise RuntimeError("All free models and providers were rate-limited or unavailable.")

# Test the rotation
response = generate_with_fallback("Testing the Harness")
print("\nFinal Output:\n", response)

Attempting model: groq -> openai/gpt-oss-20b...

Final Output:
 Hello! I’m here and ready to help. Let me know what you’d like to test or discuss.
